In [1]:
import pandas as pd

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Algorithm.ExtremeLearningMachine import ExtremeLearningMachine
from Pipeline.Algorithm.EvaluationMatrix import EvaluationMatrix
from Pipeline.Global.GlobalSetting import GlobalSetting

In [2]:
model_configs = GlobalSetting.get_model_configs()

In [3]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()

features_size = gallstone_dataset.x_train_scaled.shape[1]

In [4]:
x_test_scaled   = gallstone_dataset.x_test_scaled
y_test          = gallstone_dataset.y_test
x_train_scaled  = gallstone_dataset.x_train_scaled
y_train         = gallstone_dataset.y_train

In [5]:
testing_results = []
for config in model_configs:
    for seed in GlobalSetting.elm_testing_state_range:
        elm = ExtremeLearningMachine(
            features_size   = features_size,
            hidden_size     = config["Hidden_Nodes"],
            activation_function     = config["Activation"],
            regularization_lambda   = config["Lambda_Value"]
        )
        elm.initialize_random_weights(random_seed = seed)

        elm.fit(x_train_scaled.values, y_train.values)
        y_pred = elm.predict(x_test_scaled.values)

        evaluation = EvaluationMatrix(y_test, y_pred)
        print(f"--- Config: {config['Hidden_Nodes']} Nodes | Seed: {seed} ---")
        print(evaluation.get_report())

        metrics = evaluation.get_all_metrics()
        test_result = {
            "Model_Type"   : config.get('Model_Types', 'Unknown'),
            "Hidden_Nodes" : config['Hidden_Nodes'],
            "Lambda_Value" : config['Lambda_Value'],
            "Activation"   : config['Activation'].__name__,
            "Seed"         : seed
        }
        test_result.update(metrics)
        testing_results.append(test_result)
        print("\n")

--- Config: 44 Nodes | Seed: 81 ---
{'Counts': {'TP': np.int64(24), 'TN': np.int64(25), 'FP': np.int64(7), 'FN': np.int64(8)}, 'Metrics': {'Accuracy': np.float64(0.7656), 'Precision': 0.7742, 'Recall': 0.75, 'NPV': 0.7576, 'Specificity': 0.7812, 'F1-Score': 0.7619, 'F2-Score': 0.7547, 'Bal Accuracy': 0.7656, 'MCC': 0.5315}}


--- Config: 44 Nodes | Seed: 82 ---
{'Counts': {'TP': np.int64(23), 'TN': np.int64(24), 'FP': np.int64(8), 'FN': np.int64(9)}, 'Metrics': {'Accuracy': np.float64(0.7344), 'Precision': 0.7419, 'Recall': 0.7188, 'NPV': 0.7273, 'Specificity': 0.75, 'F1-Score': 0.7302, 'F2-Score': 0.7233, 'Bal Accuracy': 0.7344, 'MCC': 0.469}}


--- Config: 44 Nodes | Seed: 83 ---
{'Counts': {'TP': np.int64(24), 'TN': np.int64(22), 'FP': np.int64(10), 'FN': np.int64(8)}, 'Metrics': {'Accuracy': np.float64(0.7188), 'Precision': 0.7059, 'Recall': 0.75, 'NPV': 0.7333, 'Specificity': 0.6875, 'F1-Score': 0.7273, 'F2-Score': 0.7407, 'Bal Accuracy': 0.7188, 'MCC': 0.4384}}


--- Config: 44 N

In [6]:
# 1. 将字典列表转换为 DataFrame
df_results = pd.DataFrame(testing_results)

# 2. 定义哪些列代表一个唯一的 "Algorithm" 配置
group_cols = ["Model_Type", "Hidden_Nodes", "Lambda_Value", "Activation"]

# 3. 动态提取所有需要聚合的指标列（排除配置列和 Seed）
metric_cols = [col for col in df_results.columns if col not in group_cols + ["Seed"]]

# 4. 构造聚合字典：对每个指标求平均值(mean)和标准差(std)
agg_funcs = {metric: ['mean', 'std', 'max' , 'min'] for metric in metric_cols}

# 5. 执行 Group By
summary_df = df_results.groupby(group_cols).agg(agg_funcs).reset_index()

# 将多级表头 (例如 'Accuracy', 'mean') 拼接为单级表头 'Accuracy_mean'
summary_df.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in summary_df.columns.values
]

summary_df

,Model_Type,Hidden_Nodes,Lambda_Value,Activation,Accuracy_mean,Accuracy_std,Accuracy_max,Accuracy_min,Precision_mean,Precision_std,...,F2-Score_max,F2-Score_min,Bal Accuracy_mean,Bal Accuracy_std,Bal Accuracy_max,Bal Accuracy_min,MCC_mean,MCC_std,MCC_max,MCC_min
0,Best_Hidden_Nodes,44,0.000000,sigmoid,0.753125,0.030055,0.796875,0.71875,0.767231,0.043667,...,0.786164,0.681818,0.753125,0.030055,0.796875,0.71875,0.508383,0.059851,0.594040,0.438357
1,Best_Lambda,44,0.000122,sigmoid,0.753125,0.030055,0.796875,0.71875,0.767231,0.043667,...,0.786164,0.681818,0.753125,0.030055,0.796875,0.71875,0.508383,0.059851,0.594040,0.438357
2,Best_Size_and_Lambda,44,0.000122,sigmoid,0.753125,0.030055,0.796875,0.71875,0.767231,0.043667,...,0.786164,0.681818,0.753125,0.030055,0.796875,0.71875,0.508383,0.059851,0.594040,0.438357
3,Grid_Combination,44,0.000122,sigmoid,0.753125,0.030055,0.796875,0.71875,0.767231,0.043667,...,0.786164,0.681818,0.753125,0.030055,0.796875,0.71875,0.508383,0.059851,0.594040,0.438357
4,Grid_Optimization,28,0.012579,sigmoid,0.781250,0.029232,0.828125,0.75000,0.784348,0.041348,...,0.843373,0.709677,0.781250,0.029232,0.828125,0.75000,0.566333,0.057603,0.656571,0.500979
